# Merge countries and oceans shapefile

Generate a shapefile with all the countries.


### Input data:
Processed in `generate_shp_countries_wb.ipynb`

**World Bank**: [Official Boundaries - Admin 0](https://datacatalogfiles.worldbank.org/ddh-published/0038272/5/DR0095371/World%20Bank%20Official%20Boundaries%20(Shapefiles)/World%20Bank%20Official%20Boundaries%20-%20Admin%200.zip)

Oceans — [Official Boundaries - Ocean Mask]()

### Input data license:

CC0 1.0 Universal (public domain)

Giulia Cigna - giulia.cigna@polito.it<br>
Romain Thomas - romain.thomas@polito.it<br>
2026

In [1]:
import os
from dotenv import load_dotenv
import geopandas as gpd
import pandas as pd
import logging
from pathlib import Path

## LOGGING

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

## SETTINGS

In [3]:
if not os.path.exists(".env"):
    raise ValueError("You must create the '.env' file and set the values before running this notebook.")

load_dotenv()

True

## OUTPUT SETTINGS

In [4]:
# Get full shapefile path from environment
output_file = os.getenv("WB_COUNTRIES_OCEANS_PATH")
if output_file is None:
    raise ValueError("WB_COUNTRIES_OCEANS_PATH environment variable is not set")

full_path = Path(output_file)

# Create parent directory if it doesn't exist
full_path.parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Output path: {full_path}")


2026-03-18 13:48:27 - root - INFO - Output path: data/wb_oceans_countries/wb_oceans_countries.shp


## INPUT FILES

In [5]:
countries_from_wb_path = os.getenv("COUNTRIES_FROM_WB_PATH")
if countries_from_wb_path is None:
    raise ValueError("COUNTRIES_FROM_WB_PATH environment variable is not set")

oceans_from_wb_path = os.getenv("OCEANS_FROM_WB_PATH")
if oceans_from_wb_path is None:
    raise ValueError("OCEANS_FROM_WB_PATH environment variable is not set")

## READING INPUT FILES

In [6]:
# countries
logging.info(f"Reading countries shapefile  from {countries_from_wb_path}")
gdf_countries = gpd.read_file(countries_from_wb_path)

# oceans
logging.info(f"Reading ocean shapefile from {oceans_from_wb_path}")
gdf_oceans = gpd.read_file(oceans_from_wb_path)

2026-03-18 13:48:27 - root - INFO - Reading countries shapefile  from data/countries_from_wb/countries_from_wb.shp
2026-03-18 13:48:27 - root - INFO - Reading ocean shapefile from data/ocean_from_wb/ocean_from_wb.shp


## ALIGNMENT CHECK

In [7]:
# Verify columns alignment

if gdf_countries.columns.all() == gdf_oceans.columns.all():
    logging.info("Columns aligned")
else:
    logging.info("Columns system not aligned")


# Verify reference system alignment
logging.info(f"Countries reference system: {gdf_countries.crs}")
logging.info(f"Oceans reference system: {gdf_oceans.crs}")


if gdf_countries.crs == gdf_oceans.crs:
    logging.info("Reference system aligned")
else:
    logging.info("Reference system not aligned")

2026-03-18 13:48:32 - root - INFO - Columns aligned
2026-03-18 13:48:32 - root - INFO - Countries reference system: EPSG:4326
2026-03-18 13:48:32 - root - INFO - Oceans reference system: EPSG:4326
2026-03-18 13:48:32 - root - INFO - Reference system aligned


## MERGING

In [8]:
# Merge together the gdfs
gdf_final = gpd.GeoDataFrame(
    pd.concat([gdf_countries, gdf_oceans], ignore_index=True),
    crs="EPSG:4326"
)

## SAVE OUTPUT

In [9]:
# Save GeoDataFrame
gdf_final.to_file(full_path)

logging.info(f"Saved Shapefile to: {full_path}")

2026-03-18 13:48:33 - pyogrio._io - INFO - Created 255 records
2026-03-18 13:48:33 - root - INFO - Saved Shapefile to: data/wb_oceans_countries/wb_oceans_countries.shp
